In [1]:
import scanpy as sc
import omicverse as ov
import pandas as pd
ov.plot_set()


   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

Version: 1.6.11, Tutorials: https://omicverse.readthedocs.io/
Dependency error: The 'phate>=1.0' distribution was not found and is required by the application


In [2]:
adata = sc.read("/home/lugli/spuccio/Projects/SP039/GBmap/Neftel2019_Part2.h5ad")

In [3]:
adata = adata[adata.obs['donor_id'].isin(["MGH102", "MGH105", "MGH114", "MGH115", "MGH118", "MGH124", "MGH125", "MGH126", "MGH143", "MGH101", "MGH100", "MGH104", "MGH106", "MGH110", "MGH113", "MGH121", "MGH122", "MGH128", "MGH129", "MGH136", "MGH151", "MGH152", "MGH66"])]

In [4]:
df_obs = pd.DataFrame(adata.obs)

In [5]:
del adata.obs

In [8]:
adata = adata.raw.to_adata()

In [9]:
adata

AnnData object with n_obs × n_vars = 17414 × 20947
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'highly_variable_rank', 'highly_variable_features'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [9]:
adata = adata.raw.to_adata()

In [10]:
X_counts_recovered, size_factors_sub=ov.pp.recover_counts(adata.X, 50*1e4, 50*1e5, log_base=None, 
                                                          chunk_size=10000)


100%|██████████| 7414/7414 [00:12<00:00, 601.11it/s]


In [11]:
adata.X = X_counts_recovered

In [13]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["external_gene_name","ensembl_gene_id", "start_position", "end_position", "chromosome_name",],
).set_index("external_gene_name")

In [14]:
annot

,ensembl_gene_id,start_position,end_position,chromosome_name
external_gene_name,,,,
MT-TF,ENSG00000210049,577,647,MT
MT-RNR1,ENSG00000211459,648,1601,MT
MT-TV,ENSG00000210077,1602,1670,MT
MT-RNR2,ENSG00000210082,1671,3229,MT
MT-TL1,ENSG00000209082,3230,3304,MT
...,...,...,...,...
SCMH1-DT,ENSG00000235358,41241772,41338644,1
LINC01740,ENSG00000228067,212467563,212556085,1
SLC44A3-AS1,ENSG00000293271,94585556,94855426,1


In [15]:
adata.var.columns

Index(['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances', 'highly_variable_rank',
       'highly_variable_features'],
      dtype='object')

In [16]:
adata.var = adata.var[['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances']]

In [17]:
adata.var 

,mt,n_cells,percent_cells,robust,means,variances,residual_variances
feature_name,,,,,,,
ZNF470-DT,False,240,1.378201,True,0.007337,0.005306,0.828103
AC092667.2,False,32,0.183760,True,0.000962,0.000677,0.698615
ZNF367,False,788,4.525095,True,0.014325,0.007913,0.524777
SULT1B1,False,62,0.356035,True,0.001360,0.000683,0.538568
TRIM63,False,28,0.160790,True,0.000687,0.000530,0.830760
...,...,...,...,...,...,...,...
LINC01869,False,35,0.200988,True,0.000670,0.000375,0.495050
LINC02603,False,561,3.221546,True,0.012007,0.008188,0.599410
TGIF2-RAB5IF,False,487,2.796600,True,0.009415,0.005722,0.528056


In [18]:
df_tmp = pd.merge(adata.var , annot, left_index=True, right_index=True, how='left')

In [19]:
df_tmp = df_tmp.reset_index().drop_duplicates(['feature_name']).set_index(['feature_name'])

In [20]:
adata.var = df_tmp

In [21]:
adata = adata[:,adata.var['chromosome_name'].isin(["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20","21","22","X","Y","MT"])]

In [22]:
#adata.obs['author'] = df_obs['author']

In [23]:
adata.obs['donor_id'] = df_obs['donor_id']

In [24]:
import pandas as pd

# Create a DataFrame with the new metadata
metadata_data = {
    'Author': ['Neftel2019'] * 23,
    'donor_id': ["MGH102", "MGH105", "MGH114", "MGH115", "MGH118", "MGH124", "MGH125", "MGH126", "MGH143", 
                 "MGH101", "MGH100", "MGH104", "MGH106", "MGH110", "MGH113", "MGH121", "MGH122", "MGH128", 
                 "MGH129", "MGH136", "MGH151", "MGH152", "MGH66"],
    'stage': ["Primary"] * 23,
    'assay': ['10x 3\' v2', '10x 3\' v2', '10x 3\' v2', '10x 3\' v2', '10x 3\' v2', '10x 3\' v2', '10x 3\' v2', '10x 3\' v2', 
              '10x 3\' v2', 'Smart-seq2', 'Smart-seq2', 'Smart-seq2', 'Smart-seq2', 'Smart-seq2', 'Smart-seq2', 'Smart-seq2', 
              'Smart-seq2', 'Smart-seq2', 'Smart-seq2', 'Smart-seq2', 'Smart-seq2', 'Smart-seq2', 'Smart-seq2'],
    'tissue': ["left temporal lobe", "right temporal lobe", "brain", "right frontal lobe", "brain", "right frontal lobe", 
               "forebrain", "right temporal lobe", "left frontal lobe", "left frontal lobe", "left frontal lobe", 
               "right parietal lobe", "temporoparietal junction", "right frontal lobe", "right temporal lobe", 
               "left frontal lobe", "left frontal lobe", "right temporal lobe", "left temporal lobe", 
               "right parietal lobe", "left parietal lobe", "left frontal lobe", "left frontal lobe"],
    'Cells': ['Total'] * 23,
    'Method': ['cell'] * 23
}

metadata_df = pd.DataFrame(metadata_data)

# Display the metadata DataFrame
print(metadata_df)


        Author donor_id    stage       assay                    tissue  Cells  \
0   Neftel2019   MGH102  Primary   10x 3' v2        left temporal lobe  Total   
1   Neftel2019   MGH105  Primary   10x 3' v2       right temporal lobe  Total   
2   Neftel2019   MGH114  Primary   10x 3' v2                     brain  Total   
3   Neftel2019   MGH115  Primary   10x 3' v2        right frontal lobe  Total   
4   Neftel2019   MGH118  Primary   10x 3' v2                     brain  Total   
5   Neftel2019   MGH124  Primary   10x 3' v2        right frontal lobe  Total   
6   Neftel2019   MGH125  Primary   10x 3' v2                 forebrain  Total   
7   Neftel2019   MGH126  Primary   10x 3' v2       right temporal lobe  Total   
8   Neftel2019   MGH143  Primary   10x 3' v2         left frontal lobe  Total   
9   Neftel2019   MGH101  Primary  Smart-seq2         left frontal lobe  Total   
10  Neftel2019   MGH100  Primary  Smart-seq2         left frontal lobe  Total   
11  Neftel2019   MGH104  Pri

In [25]:
merged_obs_df = pd.merge(pd.DataFrame(adata.obs), metadata_df, left_on='donor_id', right_on='donor_id', how='left')

# Display the merged dataframe
print(merged_obs_df)

      donor_id      Author    stage       assay              tissue  Cells  \
0       MGH102  Neftel2019  Primary   10x 3' v2  left temporal lobe  Total   
1       MGH102  Neftel2019  Primary   10x 3' v2  left temporal lobe  Total   
2       MGH102  Neftel2019  Primary   10x 3' v2  left temporal lobe  Total   
3       MGH102  Neftel2019  Primary   10x 3' v2  left temporal lobe  Total   
4       MGH102  Neftel2019  Primary   10x 3' v2  left temporal lobe  Total   
...        ...         ...      ...         ...                 ...    ...   
17409    MGH66  Neftel2019  Primary  Smart-seq2   left frontal lobe  Total   
17410    MGH66  Neftel2019  Primary  Smart-seq2   left frontal lobe  Total   
17411    MGH66  Neftel2019  Primary  Smart-seq2   left frontal lobe  Total   
17412    MGH66  Neftel2019  Primary  Smart-seq2   left frontal lobe  Total   
17413    MGH66  Neftel2019  Primary  Smart-seq2   left frontal lobe  Total   

      Method  
0       cell  
1       cell  
2       cell  
3  

In [26]:
df_obs = df_obs[['donor_id','n_genes','nUMIs','annotation_level_1', 'annotation_level_2','annotation_level_3','scsa_celltype_cellmarker', 'scsa_celltype_panglaodb','cell_type']]

In [27]:
df_obs

,donor_id,n_genes,nUMIs,annotation_level_1,annotation_level_2,annotation_level_3,scsa_celltype_cellmarker,scsa_celltype_panglaodb,cell_type
102_1-0,MGH102,3097,2064.647705,Neoplastic,Stem-like,OPC-like,Neuron,Neurons,malignant cell
102_2-0,MGH102,2368,2073.752686,Neoplastic,Stem-like,NPC-like,Neuron,Neurons,malignant cell
102_4-0,MGH102,1223,1591.808716,Neoplastic,Stem-like,NPC-like,Neuron,Neurons,malignant cell
102_5-0,MGH102,2282,1908.673950,Neoplastic,Stem-like,OPC-like,Neuron,Neurons,malignant cell
102_7-0,MGH102,1721,1752.211670,Neoplastic,Stem-like,NPC-like,Neuron,Neurons,malignant cell
...,...,...,...,...,...,...,...,...,...
MGH66-P08-H06-0,MGH66,7103,2322.800781,Neoplastic,Differentiated-like,AC-like,Astrocyte,Interneurons,malignant cell
MGH66-P08-H07-0,MGH66,4970,2179.734619,Neoplastic,Differentiated-like,AC-like,Astrocyte,Interneurons,malignant cell
MGH66-P08-H08-0,MGH66,6618,2576.393555,Neoplastic,Stem-like,OPC-like,Astrocyte,Interneurons,malignant cell
MGH66-P08-H10-0,MGH66,4734,2219.407959,Neoplastic,Differentiated-like,MES-like,Astrocyte,Interneurons,malignant cell


In [28]:
merged_obs_df.index= df_obs.index

In [29]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method'], dtype='object')

In [30]:
merged_obs_df = pd.merge(merged_obs_df, df_obs,right_index=True,left_index=True, how='left')

In [31]:
merged_obs_df.columns

Index(['donor_id_x', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'donor_id_y', 'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [32]:
del merged_obs_df['donor_id_y']

In [33]:
merged_obs_df.columns = ['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
                         'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type']

In [34]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'n_genes', 'nUMIs', 'annotation_level_1', 'annotation_level_2',
       'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [35]:
adata.obs = merged_obs_df

In [36]:
ov.pp.score_genes_cell_cycle(adata,species='human')

calculating cell cycle phase
computing score 'S_score'
    finished: added
    'S_score', score of gene set (adata.obs).
    642 total control genes are used. (0:00:01)
computing score 'G2M_score'
    finished: added
    'G2M_score', score of gene set (adata.obs).
    643 total control genes are used. (0:00:01)
-->     'phase', cell cycle phase (adata.obs)


In [38]:
adata.write("/home/lugli/spuccio/Projects/SP039/GBmap/Neftel2019_Part3.h5ad")